<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan perhitungan fitur density

In [9]:
# Memilih dataset dan parameter pemisahan
import pandas as pd
from IPython.display import display, Markdown
from pci_ml.data import list_datasets, load_data, clean_data, split_data
from pci_ml.memory import configure, release
from pci_ml.ui import build_panel

EXPORT_MODE = True   # True = print panel values instead of showing widgets
RELEASE_MEMORY = True

DEFAULTS = {
    "file": None,        # None = first dataset, or e.g. "data_2026.xlsx"
    "n_seed": 200,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if DEFAULTS["file"] is not None and DEFAULTS["file"] not in datasets:
    raise ValueError(f"DEFAULTS['file'] '{DEFAULTS['file']}' not found in datasets")

# Helper function: compact preview of the first rows
def format_adaptive_df(df_input):
    if df_input is None:
        return ""
    df_head = df_input.head()
    left_count = 5
    right_count = 2
    if df_head.shape[1] > (left_count + right_count):
        left_cols = df_head.iloc[:, :left_count]
        right_cols = df_head.iloc[:, -right_count:]
        middle_col = pd.DataFrame({"…": ["…"] * len(df_head)}, index=df_head.index)
        df_display = pd.concat([left_cols, middle_col, right_cols], axis=1)
    else:
        df_display = df_head
    markdown_table = df_display.to_markdown(index=False)
    return f'<div style="font-size: 11px;">\n\n{markdown_table}\n\n</div>'

df = None
handle = None

def on_select(name):
    raw = None if name is None else load_data(name)
    handle.update(Markdown(format_adaptive_df(raw)))

panel = build_panel(datasets, DEFAULTS, on_file_change=on_select, export=EXPORT_MODE)

handle = display(Markdown(""), display_id=True)

on_select(panel.values()["file"])


File terpilih untuk analisis: pci-ml-26.xlsx
Parameter Pemisahan Data:
- Jumlah seed     : 200
- Test size       : 0.2
- Share tolerance : 0.01
- PCI minimum     : 0.0


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   SEGMENT_AREA |   GATOR_CRACK_L | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|---------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |            450 |            0    | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |            450 |            0    | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |            450 |            0    | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |            450 |            0.64 | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |            450 |            0    | …   |             0 | 96.1874 |

</div>

In [10]:
# Membersihkan data dari duplikat, pencilan dan data tidak valid, serta menghitung fitur density
cfg = panel.values()
filename = cfg["file"]
df, features = clean_data(load_data(filename))

if not EXPORT_MODE:
    print(f"Parameter: file {filename} | jumlah seed {cfg['n_seed']} | "
          f"test size {cfg['test_size']} | toleransi {cfg['share_tol']} | "
          f"PCI minimum {cfg['pci_min']}")

markdown_table = format_adaptive_df(df)
print("Data telah dibersihkan dan fitur densitas telah dihitung.")
display(Markdown(markdown_table))


Data telah dibersihkan dan fitur densitas telah dihitung.


<div style="font-size: 11px;">

| ROAD_NAME    | ROAD_ID   | SEGMENT_ID   |   GATOR_CRACK_L |   GATOR_CRACK_M | …   |   RAVELLING_H |     PCI |
|:-------------|:----------|:-------------|----------------:|----------------:|:----|--------------:|--------:|
| Guru Bangkol | R001      | S001         |      0          |      0          | …   |             0 | 85.4969 |
| Guru Bangkol | R001      | S002         |      0          |      0.00391778 | …   |             0 | 73.3368 |
| Guru Bangkol | R001      | S003         |      0          |      0          | …   |             0 | 89.9305 |
| Guru Bangkol | R001      | S004         |      0.00142222 |      0.0231111  | …   |             0 | 73.6287 |
| Guru Bangkol | R001      | S005         |      0          |      0          | …   |             0 | 96.1874 |

</div>

#### 2. Pemisahan data latih dan data uji

In [11]:
# Memisah data sesuai kriteria
from IPython.display import clear_output

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))

def show_progress(done, total, best_gap):
    if done % 100 != 0 and done != total:
        return
    gap = "-" if best_gap is None else f"{best_gap:.4f}"
    clear_output(wait=True)
    print(f"Mencari Seeds: {done}/{total} | Best Gap: {gap}", flush=True)

result = split_data(df, features, filename, {
    "n_seed": cfg["n_seed"],
    "test_size": cfg["test_size"],
    "share_tol": cfg["share_tol"],
    "pci_min": cfg["pci_min"],
}, progress=show_progress)
print()

split_summary = result["summary"]

# Print the summary outside the progress line so it always shows up
print(split_summary)

release("result", "df", "show_progress")


Mencari Seeds: 200/200 | Best Gap: 0.7556

Seed used: 179 (Gap: 0.7556)
Train: 933 rows (79.27%) | Test: 244 rows (20.73%)
Roads: 27 train | 7 test
Saved to: "C:\Users\gearb\Desktop\New folder\pci-ml\datasets\pci-ml-26\n200_t0.2_s0.01_p0.0"


#### 3. Pelatihan model *machine learning*

In [12]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import PCI, ROAD_ID, METADATA, latest_checkpoint, load_checkpoint

SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

X_train = X_test = y_train = y_test = groups_train = test_ids = meta = None

def load_latest_ckpt(tag):
    global X_train, X_test, y_train, y_test, groups_train, test_ids, features, meta
    # train_df / test_df are local: they are freed when this function returns
    train_df, test_df, features, meta = load_checkpoint(filename, tag)

    X_train = train_df[features].to_numpy(dtype="float32")
    X_test  = test_df[features].to_numpy(dtype="float32")
    y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
    y_test  = test_df[PCI].to_numpy(dtype="float32")

    groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
    test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)   # labels for result tables

    assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

def split_info():
    n_tr, n_te = len(X_train), len(X_test)
    total = n_tr + n_te
    return (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
            f"**Test:** {X_test.shape} ({n_te / total:.2%})")

load_latest_ckpt(ckpt_tag)

# Staleness guard: the newest split may have been made with different parameters
current = {key: cfg[key] for key in ("n_seed", "test_size", "share_tol", "pci_min")}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_info()}"))

release("df", "ckpt_tag", "load_latest_ckpt", "split_info", "current")


**Split terpilih:** `n200_t0.2_s0.01_p0.0`

**Train:** (933, 55) (79.27%) | **Test:** (244, 55) (20.73%)

In [13]:
# Verifikasi data input
import numpy as np
from pci_ml.data import ROAD_ID

# 1. Shapes and dtypes
print("Arrays:")
for name in ("X_train", "X_test", "y_train", "y_test", "groups_train"):
    a = globals()[name]
    print(f"  {name:<14}{str(a.shape):<12}{a.dtype}")
print(f"  {'test_ids':<14}{str(test_ids.shape):<12}columns: {list(test_ids.columns)}")
print(f"  {'features':<14}{len(features)} columns | first 3: {features[:3]}")

# 2. Consistency checks
checks = {
    "X columns == len(features)":        X_train.shape[1] == X_test.shape[1] == len(features),
    "X/y/groups rows match (train)":     len(X_train) == len(y_train) == len(groups_train),
    "X/y/test_ids rows match (test)":    len(X_test) == len(y_test) == len(test_ids),
    "row counts match meta":             (len(X_train), len(X_test)) == (meta["n_train"], meta["n_test"]),
    "no NaN/inf in arrays":              all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)),
    "X within 0-1":                      min(X_train.min(), X_test.min()) >= 0 and max(X_train.max(), X_test.max()) <= 1,
    "y within 0-100":                    min(y_train.min(), y_test.min()) >= 0 and max(y_train.max(), y_test.max()) <= 100,
    "no road shared by train and test":  set(groups_train).isdisjoint(test_ids[ROAD_ID]),
    "train_df/test_df released":         "train_df" not in globals() and "test_df" not in globals(),
}
print("\nChecks:")
for label, ok in checks.items():
    print(f"  [{'OK' if ok else 'FAIL'}] {label}")

# 3. Summary of the selected split
print(f"\nSeed split: {meta['seed']} | Gap: {meta['gap']:.4f} | Dibuat: {meta['created_at']}")
print(f"PCI train: mean {y_train.mean():.2f}, std {y_train.std(ddof=1):.2f} | "
      f"test: mean {y_test.mean():.2f}, std {y_test.std(ddof=1):.2f}")

Arrays:
  X_train       (933, 55)   float32
  X_test        (244, 55)   float32
  y_train       (933,)      float32
  y_test        (244,)      float32
  groups_train  (933,)      object
  test_ids      (244, 3)    columns: ['ROAD_ID', 'ROAD_NAME', 'SEGMENT_ID']
  features      55 columns | first 3: ['GATOR_CRACK_L', 'GATOR_CRACK_M', 'GATOR_CRACK_H']

Checks:
  [OK] X columns == len(features)
  [OK] X/y/groups rows match (train)
  [OK] X/y/test_ids rows match (test)
  [OK] row counts match meta
  [OK] no NaN/inf in arrays
  [OK] X within 0-1
  [OK] y within 0-100
  [OK] no road shared by train and test
  [OK] train_df/test_df released

Seed split: 179 | Gap: 0.7556 | Dibuat: 2026-11-07 | 17:21:06
PCI train: mean 68.37, std 22.92 | test: mean 68.32, std 22.20


In [ ]:
# Melatih dan menguji model Random Forest
from pci_ml.random_forest import train_random_forest, evaluate_random_forest
from pci_ml.ui import build_param_panel

RF_DEFAULTS = {
    "n_estimators": 300,
    "max_depth": 0,          # 0 = kedalaman tidak dibatasi
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 1.0,
    "max_samples": 1.0,      # 1.0 = semua sampel
}

RF_SPECS = {
    "n_estimators":      {"type": "int",   "description": "Jumlah pohon:", "label": "Jumlah pohon"},
    "max_depth":         {"type": "int",   "description": "Kedalaman maks:", "label": "Kedalaman maks"},
    "min_samples_split": {"type": "int",   "description": "Min sampel split:", "label": "Min sampel split"},
    "min_samples_leaf":  {"type": "int",   "description": "Min sampel daun:", "label": "Min sampel daun"},
    "max_features":      {"type": "float", "description": "Maks fitur:", "label": "Maks fitur"},
    "max_samples":       {"type": "float", "description": "Maks sampel:", "label": "Maks sampel"},
}

rf_panel = build_param_panel(RF_DEFAULTS, RF_SPECS, "Parameter Random Forest:", export=EXPORT_MODE)
rf_params = rf_panel.values()

if rf_params["max_depth"] == 0:
    print("Catatan: kedalaman 0 berarti tanpa batas.")

model = train_random_forest(X_train, y_train, rf_params, random_state=SEED)
train_metrics = evaluate_random_forest(model, X_train, y_train)
test_metrics = evaluate_random_forest(model, X_test, y_test)

print("\nHasil Random Forest:")
print(f"- Data latih: MAE {train_metrics['mae']:.4f} | RMSE {train_metrics['rmse']:.4f} | R² {train_metrics['r2']:.4f}")
print(f"- Data uji  : MAE {test_metrics['mae']:.4f} | RMSE {test_metrics['rmse']:.4f} | R² {test_metrics['r2']:.4f}")


In [ ]:
# Tuning hyperparameter Random Forest dengan GroupKFold
from pci_ml.random_forest import tune_random_forest, train_random_forest, evaluate_random_forest
from pci_ml.ui import build_param_panel

TUNING_DEFAULTS = {
    "n_iter": 60,
    "cv_splits": 5,
}

TUNING_SPECS = {
    "n_iter":    {"type": "int", "description": "Jumlah kandidat:", "label": "Jumlah kandidat"},
    "cv_splits": {"type": "int", "description": "Jumlah fold:", "label": "Jumlah fold"},
}

RF_SEARCH_SPACE = {
    "n_estimators": [300, 500],
    "max_depth": [None, 40, 30, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 3],
    "max_features": [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0],
    "max_samples": [None, 0.6, 0.7, 0.8, 0.9],
}

tuning_panel = build_param_panel(
    TUNING_DEFAULTS, TUNING_SPECS, "Parameter Tuning Random Forest:", export=EXPORT_MODE
)
tuning_params = tuning_panel.values()

print("Mencari hyperparameter Random Forest...")
rf_search = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=tuning_params["n_iter"],
    cv_splits=tuning_params["cv_splits"],
    random_state=SEED,
)

rf_best_params = rf_search.best_params_
rf_best_cv_mae = -rf_search.best_score_
tuned_model = train_random_forest(X_train, y_train, rf_best_params, random_state=SEED)
tuned_train = evaluate_random_forest(tuned_model, X_train, y_train)
tuned_test = evaluate_random_forest(tuned_model, X_test, y_test)

print("\nHyperparameter terbaik:")
for key in RF_SEARCH_SPACE:
    value = rf_best_params[key]
    if value is None:
        value = "tanpa batas"
    print(f"- {key:<18}: {value}")
print(f"- CV MAE            : {rf_best_cv_mae:.4f}")

print("\nHasil Random Forest setelah tuning:")
print(f"- Data latih: MAE {tuned_train['mae']:.4f} | RMSE {tuned_train['rmse']:.4f} | R² {tuned_train['r2']:.4f}")
print(f"- Data uji  : MAE {tuned_test['mae']:.4f} | RMSE {tuned_test['rmse']:.4f} | R² {tuned_test['r2']:.4f}")

baseline_gap = train_metrics["r2"] - test_metrics["r2"]
tuned_gap = tuned_train["r2"] - tuned_test["r2"]
print("\nPerbandingan dengan baseline:")
print(f"- MAE uji            : {test_metrics['mae']:.4f} -> {tuned_test['mae']:.4f}")
print(f"- R² uji             : {test_metrics['r2']:.4f} -> {tuned_test['r2']:.4f}")
print(f"- Gap R² (latih-uji) : {baseline_gap:.4f} -> {tuned_gap:.4f}")


### ***Source code* modul dan fungsi**

In [ ]:
# import os
# from pathlib import Path
# from IPython.display import display, Markdown

# # Path to your src directory
# src_dir = Path("src")

# # Loop through all Python files and render them as Markdown code blocks
# for filepath in src_dir.rglob("*.py"): # rglob searches subdirectories too
    
#     # Ignore any files inside Jupyter checkpoint folders
#     if ".ipynb_checkpoints" in filepath.parts:
#         continue

#     with open(filepath, "r", encoding="utf-8") as f:
#         code = f.read()
    
#     # Creates a nice header for the filename and formats the code
#     display(Markdown(f"#### `{filepath.name}`\n```python\n{code}\n```\n---"))

#### `data.py`
```python
import os
import json
from datetime import datetime
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from pathlib import Path

data_dir = Path(__file__).resolve().parents[2] / "datasets"
PCI, ROAD_ID, AREA = "PCI", "ROAD_ID", "SEGMENT_AREA"
METADATA = ["ROAD_NAME", "SEGMENT_ID"]

def list_datasets():
    return sorted (
        f for f in os.listdir(data_dir)
        if os.path.isfile(os.path.join(data_dir, f))
        and f.lower().endswith((".xlsx", ".xls"))
        and not f.startswith("~$")
    )

def load_data(filename):
    path = os.path.join(data_dir, filename)
    if filename.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    return pd.read_csv(path)

def clean_data(df):
    excluded = {PCI, ROAD_ID, AREA, *METADATA}
    features = [c for c in df.columns if c not in excluded]

    df = df.drop_duplicates().copy()
    df[PCI] = df[PCI].clip(lower=0, upper=100)
    df[features] = df[features].div(df[AREA], axis=0).clip(lower=0, upper=1)
    df = df.drop(columns=[AREA])
    return df, features

def seed_search(df, features, params, progress):
    n_seed = int(params["n_seed"])
    test_size = params["test_size"]
    tol = params["share_tol"]
    pci_min = params["pci_min"]

    observed = [c for c in features if (df[c] != 0).any()]
    nonzero = (df[observed] != 0).to_numpy()
    pci_values = df[PCI].to_numpy()

    best_gap, best_seed, best_idx = None, None, None

    for seed in range(n_seed):
        try:
            progress(seed, n_seed, best_gap)

            gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
            train_idx, test_idx = next(gss.split(df, groups=df[ROAD_ID]))

            share = len(test_idx) / len(df)
            if not (test_size - tol) <= share <= (test_size + tol):
                continue

            if (~nonzero[train_idx].any(axis=0) & nonzero[test_idx].any(axis=0)).any():
                continue

            y_train, y_test = pci_values[train_idx], pci_values[test_idx]
            if y_test.min() > pci_min:
                continue

            gap = abs(y_train.mean() - y_test.mean()) + abs(y_train.std(ddof=1) - y_test.std(ddof=1))

            if best_gap is None or gap < best_gap:
                best_gap, best_seed, best_idx = gap, seed, (train_idx, test_idx)
        finally:
            if seed == n_seed - 1:
                progress(n_seed, n_seed, best_gap)

    if best_seed is None:
        raise ValueError("No seed passed the criteria.")
    return best_idx[0], best_idx[1], best_seed, float(best_gap)

def _tag(params):
    return (f"n{params['n_seed']}_t{params['test_size']}"
            f"_s{params['share_tol']}_p{params['pci_min']}")

def save_checkpoint(train_df, test_df, features, filename, params, seed, gap,
                    out_root=data_dir):
    out_dir = Path(out_root) / Path(filename).stem / _tag(params)
    out_dir.mkdir(parents=True, exist_ok=True)

    train_df.to_csv(out_dir / "train.csv", index=False)
    test_df.to_csv(out_dir / "test.csv", index=False)

    meta = {
        "source_file": filename,
        "params": params,
        "seed": int(seed),
        "gap": float(gap),
        "features": features,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "created_at": datetime.now().strftime("%Y-%m-%d | %H:%M:%S"),
    }
    with open(out_dir / "meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)
    return out_dir

def split_data(df, features, filename, params, progress):
    train_idx, test_idx, seed, gap = seed_search(df, features, params, progress=progress)
    train_df = df.iloc[train_idx].copy()
    test_df = df.iloc[test_idx].copy()

    out_dir = save_checkpoint(train_df, test_df, features, filename, params, seed, gap)

    summary = (
        f"Seed used: {seed} (Gap: {gap:.4f})\n"
        f"Train: {len(train_df)} rows ({len(train_df) / len(df):.2%}) | "
        f"Test: {len(test_df)} rows ({len(test_df) / len(df):.2%})\n"
        f"Roads: {train_df[ROAD_ID].nunique()} train | {test_df[ROAD_ID].nunique()} test\n"
        f'Saved to: "{out_dir}"'
    )
    return {"train": train_df, "test": test_df, "features": features,
            "seed": seed, "gap": gap, "summary": summary}

def list_checkpoints(filename, out_root=data_dir):
    base = Path(out_root) / Path(filename).stem
    if not base.exists():
        return []
    return [p.name for p in base.iterdir() if p.is_dir() and (p / "meta.json").exists()]

def load_checkpoint(filename, tag, out_root=data_dir):
    out_dir = Path(out_root) / Path(filename).stem / tag
    train_df = pd.read_csv(out_dir / "train.csv")
    test_df = pd.read_csv(out_dir / "test.csv")
    with open(out_dir / "meta.json", encoding="utf-8") as f:
        meta = json.load(f)
    return train_df, test_df, meta["features"], meta

def latest_checkpoint(filename, out_root=data_dir):
    tags = list_checkpoints(filename, out_root)
    if not tags:
        raise FileNotFoundError("run the split first")
    base = Path(out_root) / Path(filename).stem

    def created_at(tag):
        with open(base / tag / "meta.json", encoding="utf-8") as f:
            return json.load(f)["created_at"]

    return max(tags, key=created_at)

```
---

#### `memory.py`
```python
import gc

from IPython import get_ipython

# Setting, changed through configure() from the notebook
_cfg = {"release": True}


def configure(release=True):
    """release=False makes release() do nothing (useful for comparing runs)."""
    _cfg["release"] = release


def release(*names):
    """Close widgets, delete the named notebook globals, then collect garbage."""
    ip = get_ipython()
    if not _cfg["release"] or ip is None:
        return
    try:
        import ipywidgets as wg
        widget_type = wg.Widget
    except ImportError:
        widget_type = ()

    for name in names:
        if name not in ip.user_ns:
            continue
        obj = ip.user_ns[name]
        if widget_type and isinstance(obj, widget_type):
            obj.close()
        ip.del_var(name, by_name=True)
        del obj
    _ = gc.collect()
```
---

#### `ui.py`
```python
"""Control panel for the PCI ML notebook."""

import ipywidgets as wg
from IPython.display import display


def _resolve_file(datasets, defaults):
    return defaults["file"] or (datasets[0] if datasets else None)


class ControlPanel:
    """Holds the dataset dropdown and the split parameters."""

    def __init__(self, datasets, defaults, on_file_change=None):
        style = {"description_width": "110px"}
        box = wg.Layout(width="180px", margin="0 8px 0 0")

        self.file = wg.Dropdown(
            options=datasets,
            value=_resolve_file(datasets, defaults),
            description="Pilih file :",
            style=style,
            layout=wg.Layout(width="400px", margin="0 0 6px 0"),
        )
        self.n_seed = wg.IntText(value=defaults["n_seed"], description="Jumlah seed:",
                                 style=style, layout=box)
        self.test_size = wg.FloatText(value=defaults["test_size"], description="Test size:",
                                      style=style, layout=box)
        self.share_tol = wg.FloatText(value=defaults["share_tol"], description="Tolerance:",
                                      style=style, layout=box)
        self.pci_min = wg.FloatText(value=defaults["pci_min"], description="PCI minimum:",
                                    style=style, layout=box)

        self._on_file_change = on_file_change
        self.file.observe(self._file_changed, names="value")

        self.widget = wg.VBox([
            self.file,
            wg.HBox([self.n_seed, self.test_size, self.share_tol, self.pci_min],
                    layout=wg.Layout(align_items="flex-start", flex_flow="row wrap")),
        ], layout=wg.Layout(align_items="flex-start"))

    def _file_changed(self, change):
        if self._on_file_change is not None:
            self._on_file_change(change["new"])

    def values(self):
        """Return a plain snapshot of the current panel values."""
        return {
            "file": self.file.value,
            "n_seed": int(self.n_seed.value),
            "test_size": float(self.test_size.value),
            "share_tol": float(self.share_tol.value),
            "pci_min": float(self.pci_min.value),
        }


class StaticPanel:
    """Read-only panel used when export mode is on."""

    def __init__(self, values):
        self._values = values

    def values(self):
        return dict(self._values)


def build_panel(datasets, defaults, on_file_change=None, export=False):
    """Build and display the control panel; return it for snapshotting.

    In export mode the widgets are replaced by printed values taken from
    ``defaults``. The caller triggers the initial preview after creating
    its output handle, so the panel appears above that preview.
    """
    if export:
        values = {
            "file": _resolve_file(datasets, defaults),
            "n_seed": int(defaults["n_seed"]),
            "test_size": float(defaults["test_size"]),
            "share_tol": float(defaults["share_tol"]),
            "pci_min": float(defaults["pci_min"]),
        }
        print(f"File terpilih untuk analisis: {values['file']}")
        print("Parameter Pemisahan Data:")
        print(f"- Jumlah seed     : {values['n_seed']}")
        print(f"- Test size       : {values['test_size']}")
        print(f"- Share tolerance : {values['share_tol']}")
        print(f"- PCI minimum     : {values['pci_min']}")
        return StaticPanel(values)

    panel = ControlPanel(datasets, defaults, on_file_change=on_file_change)
    display(panel.widget)
    return panel

```
---

#### `__init__.py`
```python

```
---